<a href="https://colab.research.google.com/github/marcory-hub/yolo11n-on-grove-vision-ai-v2/blob/main/YOLO_training_2026_02_19.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# YOLO11n training




Last accessed: 2026-02-19 runtime 2026.01
- train with dataset vespa 2026-02 v1
- 60px, 40px, 30px minimum object size when scaled to 192px)
- 192 and 224 imgsz

1. Make sure images and labels from your dataset have this folder structure with these exact names. And add `data.yaml` to main folder.

```
🗂️ dataset
  🗂️ train
    🗂️ images
    🗂️ labels
  🗂️ valid
    🗂️ images
    🗂️ labels
  data.yaml
```

2. Zip the dataset folder to a file names `dataset.zip` On mac use `zip -r dataset.zip dataset -i '*.yaml'*.jpg' '*.txt' '*/'` to include yaml, jpg and txt only.

3. Copy the `dataset.zip` file to /`content/drive/MyDrive`, it is needed to make a callibration image set and your yolo model, fe `best.pt` to this folder. For the model you can use a custom name and adjust it in the options below.

4. Copy the `dataset.zip` file to the folder /content/drive/MyDrive/yolo.


In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
#check GPU
gpu_info = !nvidia-smi
gpu_info = '\n'.join(gpu_info)
if gpu_info.find('failed') >= 0:
  print('Not connected to a GPU')
else:
  print(gpu_info)

# Adjust the zipped file name

In [ ]:
# Copy zipped dataset to colab and unzip the dataset
!cp '/content/drive/MyDrive/dataset.zip' '/content/dataset.zip'
!unzip '/content/dataset.zip' -d '/content/dataset/'

In [ ]:
# Install the required packages for Ultralytics YOLO and Weights & Biases
!pip install -U ultralytics==8.4.14 wandb

In [ ]:
from ultralytics import YOLO

# Initialize YOLO
yolo = YOLO()


# Train, zip and download the model

Action:
1. Adjust epochs (select 10 for testing purposes, 300 for training)
2. Adust batch_size: common batch sizes that work well with GPU architectures are powers of two, such as 16, 32, 64, 128, ... 512. -1 for autobatch. (395 worked with T4-high ram)
3. Set image size (default 192, max 224).
4. remove if these aumentations were done in the dataset:             
- fliplr=0.0,
- degrees=10.0,
- hsv_h=0.0,
- hsv_s=0.0,
- hsv_v=0.0,
- epochs=epochs,
- lr0=0.005,
- warmup_epochs=10,
- patience=30,


-----

In [ ]:
import os
import shutil
from google.colab import userdata, drive
from ultralytics import YOLO, settings

drive_save_path = "/content/drive/MyDrive/YOLO_trainings"

# Enable W&B in Ultralytics
os.environ["WANDB_API_KEY"] = userdata.get('wandb-key')
settings.update({"wandb": True})

# Config
project_name = "vespa_2026-02"
name = "yolo11n_v1_e300_b395_imgsz224"

# Train
model = YOLO("yolo11n.pt")

model.train(
    data="/content/dataset/data.yaml",
    epochs=300,
    imgsz=224,
    batch=395,
    project=project_name,
    name=name,
    patience=30
)

# Backup to Drive
final_local_path = f"{project_name}/{name}"
final_drive_path = f"{drive_save_path}/{project_name}/{name}"

os.makedirs(os.path.dirname(final_drive_path), exist_ok=True)
if os.path.exists(final_local_path):
    shutil.copytree(final_local_path, final_drive_path, dirs_exist_ok=True)
    print(f"Results backed up to: {final_drive_path}")


In [ ]:
import shutil
from google.colab import files

# Paths
source_folder = "/content/runs/detect/vespa_2026-02"
zip_file = "/content/vespa_2026-02.zip"

# Create ZIP
shutil.make_archive(zip_file.replace('.zip',''), 'zip', source_folder)
print(f"✅ Folder zipped to {zip_file}")

# Download to local computer
files.download(zip_file)
